[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zjelveh/zjelveh.github.io/blob/master/files/ccjs418e_fall2026/6_groupby.ipynb)

**IMPORTANT: Save your own copy.**
1. Click File > Save a copy in Drive.
2. Rename it.
3. Work in your copy.

---

# 6. Groupby: Comparing Many Groups at Once
## CCJS 418E: Coding for Criminology

Filtering selects the records we need. Groupby lets us calculate a separate result for each group of records.

Our main question is:

> How did reported property-crime rates differ between 2015–2017 and 2018–2019 in Delaware, Maryland, Pennsylvania, and Virginia? Did motor vehicle theft follow the same pattern?

We will count records, calculate averages and totals, find minimums and maximums, and combine filtering with grouping. Then we will use our first result to guide a follow-up comparison.


## Part 1: Meet the Prepared Data

We will use the same four states and five years as the previous two notebooks. One row represents **one state in one year**.

This prepared version keeps the six familiar columns and adds `totals_property_all`, the number of reported property crimes in that state and year. The count comes from the same CORGIS State Crime source as the rates. The file is ready to analyze.


In [ ]:
import pandas as pd
import numpy as np

np.set_printoptions(legacy="1.25")

data_url = "https://raw.githubusercontent.com/zjelveh/zjelveh.github.io/refs/heads/master/files/ccjs418e_fall2026/state_crime_groupby.csv"
df = pd.read_csv(data_url)


In [ ]:
df.head(n=5)


In [ ]:
df.shape


The file contains 20 rows and seven columns.

- `state` and `year` identify the state-year record.
- `property_all`, `property_burglary`, `property_larceny`, and `property_motor` are reported offenses **per 100,000 people**.
- `totals_property_all` is a **count of reported offenses**.

Counting rows tells us how many state-years are in the file.


## Part 2: From Filtering to Grouping

We already know how to find the average property-crime rate for one state: filter to that state's records, then calculate the mean.


In [ ]:
maryland = df[df['state'] == 'Maryland']
maryland['property_all'].mean()


In [ ]:
virginia = df[df['state'] == 'Virginia']
virginia['property_all'].mean()


The calculation is the same for both states. Only the group changes. Groupby applies the same calculation to each state.

### Split, Calculate, Combine

Begin with a small table: three states observed in 2018 and 2019.


In [ ]:
demo = df[
    (df['state'].isin(['Delaware', 'Maryland', 'Virginia'])) &
    (df['year'] >= 2018)
]

demo[['state', 'year', 'property_all']]


To calculate an average for each state, pandas:

1. **Splits** the rows by the values in `state`.
2. **Calculates** the mean of `property_all` within each group.
3. **Combines** the results, with one average for each state.

For example, Delaware's group contains rates of 2,324.4 and 2,252.2. Its mean is 2,288.3 reported property crimes per 100,000 people.


In [ ]:
demo_averages = demo.groupby('state')['property_all'].mean()
demo_averages


Read the code from left to right:

- `demo`: start with the six-row table.
- `.groupby('state')`: make a group for each state.
- `['property_all']`: select the measure.
- `.mean()`: calculate its average in each group.

The result has three values: one per state.

The state names label the results. If we want a regular DataFrame with `state` as a column, we can use `.reset_index()`.


In [ ]:
demo_table = demo_averages.reset_index()
demo_table


## Part 3: Choose the Calculation That Answers the Question

### Count Records with `.size()`

How many state-year records do we have for each state?


In [ ]:
records_by_state = df.groupby('state').size()
records_by_state


Each state has five records: one state-year record for each of the five years.

### Calculate Averages with `.mean()`

What was each state's average annual property-crime rate across the five years?


In [ ]:
average_rates = df.groupby('state')['property_all'].mean()
average_rates.sort_values(ascending=False)


Each year receives equal weight in this average. The result summarizes the five annual rates.

### Add Counts with `.sum()`

How many property crimes were reported in each state across all five years? Use `totals_property_all`, which stores the reported offense count for each state-year.


In [ ]:
five_year_totals = df.groupby('state')['totals_property_all'].sum()
five_year_totals.sort_values(ascending=False)


Pennsylvania has the largest five-year count, while Delaware has the highest average annual rate in this file. Those results answer different questions. Counts describe the number of reported offenses; rates account for population size.

### Find Minimums and Maximums

What were the lowest and highest annual property-crime rates observed for each state?


In [ ]:
lowest_rates = df.groupby('state')['property_all'].min()
lowest_rates


In [ ]:
highest_rates = df.groupby('state')['property_all'].max()
highest_rates


In [ ]:
rate_ranges = highest_rates - lowest_rates
rate_ranges.sort_values(ascending=False)


Subtracting the minimum from the maximum gives the range of annual rates within each state. Pandas matches the state labels when subtracting, so Delaware's minimum is subtracted from Delaware's maximum, and so on.

A range tells us how far apart the highest and lowest values are. To investigate change over time, compare records from clearly defined periods.


## Part 4: Complete Analysis Workflow

Our question is:

> How did reported property-crime rates differ between 2015–2017 and 2018–2019 in the four states?

### Step 1: Define and Check the Two Periods

We will compare the first three years with the last two. Filter first so that every later calculation uses the intended records.


In [ ]:
earlier = df[
    (df['year'] >= 2015) &
    (df['year'] <= 2017)
]

earlier.groupby('state').size()


In [ ]:
later = df[
    (df['year'] >= 2018) &
    (df['year'] <= 2019)
]

later.groupby('state').size()


Each state has three earlier records and two later records. Calculating a mean gives us an **average annual rate** for each period.

### Step 2: Calculate a Result for Every State in Each Period


In [ ]:
earlier_property = earlier.groupby('state')['property_all'].mean()
earlier_property.round(1)


In [ ]:
later_property = later.groupby('state')['property_all'].mean()
later_property.round(1)


### Step 3: Compare the Results

Subtract the earlier average from the later average for each state:

`change = later average - earlier average`

A negative result means the later average was lower. A positive result means it was higher. The unit remains reported offenses per 100,000 people.


In [ ]:
property_change = later_property - earlier_property
property_change.round(1)


### Step 4: Answer the Main Question

The later average property-crime rate was lower in all four states. Delaware had the largest absolute decrease, about 344.2 reported offenses per 100,000 people. Virginia had the smallest, about 185.4 per 100,000.

The period averages summarize the change. Inspecting the individual annual records would show how the rate moved from year to year.


## Part 5: Use the Result to Ask a Follow-Up

The overall property-crime rate was lower in every state. Did every component follow that pattern?

Motor vehicle theft is part of property crime. We can repeat the same comparison using `property_motor`.


In [ ]:
earlier_motor = earlier.groupby('state')['property_motor'].mean()
earlier_motor.round(2)


In [ ]:
later_motor = later.groupby('state')['property_motor'].mean()
later_motor.round(2)


In [ ]:
motor_change = later_motor - earlier_motor
motor_change.round(2)


Motor vehicle theft varied across states:

- Maryland's later average was lower by about 19.72 reported offenses per 100,000 people.
- Delaware's and Virginia's later averages were higher by about 16.35 and 12.30.
- Pennsylvania's averages were almost unchanged, differing by about 0.17.

The overall property-crime result hides differences among its components. Our first result led to a closer look at one component of property crime.


## Part 6: Group by Two Columns

We can also keep the states and periods together in one table.

First, create a Boolean column. `later_period` is `False` for 2015–2017 and `True` for 2018–2019. This rule works here because the file contains only 2015–2019.


In [ ]:
df['later_period'] = df['year'] >= 2018
df[['state', 'year', 'later_period']].head(n=5)


Group by both `state` and `later_period`. Pandas makes a group for each combination, such as Delaware's earlier years and Delaware's later years. Then calculate the average property-crime rate within each combination.


In [ ]:
state_period_averages = df.groupby(
    ['state', 'later_period']
)['property_all'].mean().reset_index()

state_period_averages.round(1)


There are eight results: four states, each with two periods. Using `.reset_index()` makes both grouping variables ordinary columns, so we can inspect or filter the result using familiar methods.


In [ ]:
later_state_averages = state_period_averages[
    state_period_averages['later_period'] == True
]

later_state_averages.sort_values(by='property_all', ascending=False)


## Hands-On Exercise: Investigate Burglary Patterns

Our main analysis found lower overall property-crime rates in the later period. Now investigate burglary using the same four states and years.

Complete each step using the methods from this notebook.


In [ ]:
# Step 1: Check the records and calculate an average
# Use groupby and .size() to count the state-year records for each state.
# Calculate the mean property_burglary rate for each state.
# Display the means from highest to lowest.




In [ ]:
# Step 2: Compare totals with rates
# Sum totals_property_all for each state across the five years.
# Compare the ranking with the average_rates result from Part 3.
# Does the state with the largest count also have the highest average rate?




In [ ]:
# Step 3: Look beyond the average burglary rate
# Find the minimum and maximum property_burglary rate for each state.
# Subtract the minimums from the maximums to calculate each state's range.
# Display the ranges from largest to smallest.




In [ ]:
# Step 4: Compare burglary across the two periods
# Filter df into 2015-2017 records and 2018-2019 records.
# For each period, calculate the mean property_burglary rate for each state.
# Subtract the earlier averages from the later averages.
# Display the changes. Did burglary follow the overall property-crime pattern?




In [ ]:
# Step 5: Investigate burglary's share of property crime
# Create burglary_pct: property_burglary / property_all * 100.
# Group by state and later_period, then calculate the mean burglary_pct.
# Use .reset_index() to display the eight results as a DataFrame.
# Compare each state's earlier and later average percentages.
# Did burglary's average percentage move in the same direction as its rate?




## Reference: Groupby Patterns

```python
# Count rows in each group
df.groupby('group_column').size()

# Calculate a mean, sum, minimum, or maximum
df.groupby('group_column')['measure'].mean()
df.groupby('group_column')['count_column'].sum()
df.groupby('group_column')['measure'].min()
df.groupby('group_column')['measure'].max()

# Filter, then group
selected = df[df['year'] >= 2018]
selected.groupby('state')['property_all'].mean()

# Group by two columns and return a DataFrame
df.groupby(['state', 'later_period'])['property_all'].mean().reset_index()

# Sort a grouped Series
average_rates.sort_values(ascending=False)

# Sort a DataFrame by a named column
state_period_averages.sort_values(by='property_all', ascending=False)
```

Remember:

- `.size()` counts rows, regardless of what one row represents.
- `.sum()` adds the values in a column. Use count columns to total reported offenses.
- `.mean()` averages the selected values within each group.
- Minimums and maximums give each group's lowest and highest values.
- Two grouping columns produce one result for each combination present in the data.
- When averaging annual rates or percentages, describe the result as an average of those annual values.


## Wrap-Up: Key Takeaways

Today you learned to:

1. Replace repeated group-specific calculations with groupby.
2. Choose `.size()`, `.mean()`, `.sum()`, `.min()`, or `.max()` based on the question and the meaning of the rows and columns.
3. Filter to the intended records before grouping.
4. Compare grouped results across periods.
5. Group by two columns and use `.reset_index()` to create a regular DataFrame.
6. Develop a connected follow-up when an overall result leaves something important unanswered.

Next, we will use charts to display comparisons like these.
